# 1

In [2]:
import numpy as np
import time, math, random
import os 

# ==========================================
# 1. BUILD TARGET & F2 MATH
# ==========================================
def build_target_f2():
    T = np.zeros((9, 9, 9), dtype=np.int8)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1
    return T

T_target = build_target_f2()

def calc_f2_tensor(U, V, W):
    recon = np.einsum("ir,jr,kr->ijk", U.astype(np.int32), V.astype(np.int32), W.astype(np.int32))
    return (recon % 2).astype(np.int8)

# ==========================================
# 2. SIMULATED ANNEALING ENGINE (incremental + error-biased)
# ==========================================
def simulated_annealing_f2(U_full, V_full, W_full, col_drop,
                            iterations=25_000_000, focus_prob=0.7,
                            checkpoint_path='sa_checkpoint.npz'):
    start_t = time.time()

    keep = [i for i in range(23) if i != col_drop]
    U = U_full[:, keep].copy()
    V = V_full[:, keep].copy()
    W = W_full[:, keep].copy()
    rank = U.shape[1]

    # Initial full reconstruction (only computed once, here)
    recon = calc_f2_tensor(U, V, W)
    error_mask = (recon != T_target)
    current_error = int(error_mask.sum())

    # Live set of violated (i,j,k) positions, for O(1) biased sampling
    violated = set(map(tuple, np.argwhere(error_mask)))

    best_error = current_error
    best_U, best_V, best_W = U.copy(), V.copy(), W.copy()

    T_start, T_end = 2.0, 0.005
    cooling_factor = (T_end / T_start) ** (1.0 / iterations)
    temp = T_start

    print(f"Starting F2 Simulated Annealing | Dropping Col {col_drop}")
    print(f"Initial Errors: {current_error}/729")
    print(f"Schedule: {iterations} steps | Temp: {T_start} -> {T_end}\n")

    for i in range(iterations):
        # ---- 1. Choose which bit to flip (biased toward violated cells) ----
        if violated and random.random() < focus_prob:
            vi, vj, vk = random.choice(tuple(violated))
            r = random.randint(0, rank - 1)
            mat_choice = random.randint(0, 2)
            row = vi if mat_choice == 0 else (vj if mat_choice == 1 else vk)
            col = r
        else:
            mat_choice = random.randint(0, 2)
            row = random.randint(0, 8)
            col = random.randint(0, rank - 1)

        # ---- 2. Which (i,j,k) cells does this flip actually touch? ----
        # Flipping M[row,col] only toggles recon[i,j,k] where the OTHER
        # two matrices both have a 1 in column `col` at the relevant row.
        if mat_choice == 0:      # flip U[row, col]
            js = np.nonzero(V[:, col])[0]
            ks = np.nonzero(W[:, col])[0]
            idx_i = np.full(js.size * ks.size, row)
            idx_j = np.repeat(js, ks.size)
            idx_k = np.tile(ks, js.size)
        elif mat_choice == 1:    # flip V[row, col]
            is_ = np.nonzero(U[:, col])[0]
            ks = np.nonzero(W[:, col])[0]
            idx_i = np.repeat(is_, ks.size)
            idx_j = np.full(is_.size * ks.size, row)
            idx_k = np.tile(ks, is_.size)
        else:                     # flip W[row, col]
            is_ = np.nonzero(U[:, col])[0]
            js = np.nonzero(V[:, col])[0]
            idx_i = np.repeat(is_, js.size)
            idx_j = np.tile(js, is_.size)
            idx_k = np.full(is_.size * js.size, row)

        touched = (idx_i, idx_j, idx_k)

        # ---- 3. Compute the error delta WITHOUT recomputing the tensor ----
        old_mismatch = error_mask[touched]
        new_recon_vals = 1 - recon[touched]
        new_mismatch = (new_recon_vals != T_target[touched])
        delta_E = int(new_mismatch.sum()) - int(old_mismatch.sum())

        # ---- 4. Metropolis acceptance ----
        accept = False
        if delta_E <= 0:
            accept = True
        elif random.random() < math.exp(-delta_E / temp):
            accept = True

        # ---- 5. Apply or reject ----
        if accept:
            if mat_choice == 0:
                U[row, col] ^= 1
            elif mat_choice == 1:
                V[row, col] ^= 1
            else:
                W[row, col] ^= 1

            recon[touched] = new_recon_vals
            for (ii, jj, kk), was_wrong, is_wrong in zip(zip(*touched), old_mismatch, new_mismatch):
                cell = (int(ii), int(jj), int(kk))
                if was_wrong and not is_wrong:
                    violated.discard(cell)
                elif not was_wrong and is_wrong:
                    violated.add(cell)
            error_mask[touched] = new_mismatch
            current_error += delta_E

            if current_error < best_error:
                best_error = current_error
                best_U, best_V, best_W = U.copy(), V.copy(), W.copy()
                print(f"  Step {i:9d} | Temp: {temp:.4f} | NEW BEST: {best_error}/729 errors")
                if checkpoint_path:
                    try:
                        os.makedirs(os.path.dirname(checkpoint_path) or '.', exist_ok=True)
                        np.savez(checkpoint_path, U=best_U, V=best_V, W=best_W,
                                 col_drop=col_drop, error=best_error)
                        print(f"Saved checkpoint to {checkpoint_path}")
                    except Exception as e:
                        print(f"Failed saving checkpoint {checkpoint_path}: {e}")

            if best_error == 0:
                print(f"\n*** EXACT RANK-{rank} F2 SOLUTION FOUND AT STEP {i}! ***")
                break

        temp *= cooling_factor

        if i > 0 and i % 1_000_000 == 0:
            print(f"  [Heartbeat] Step {i:9d} | Temp: {temp:.4f} | "
                  f"Current: {current_error}/729 | Best: {best_error}/729 | "
                  f"Tracked violations: {len(violated)}")

    print(f"\nDone in {time.time()-start_t:.1f}s | Final Best Error: {best_error}/729")
    return best_error, best_U, best_V, best_W


# ==========================================
# 3. EXECUTION BLOCK (Robust Loader)
# ==========================================
if __name__ == "__main__":
    print("Loading F2 Factorizations...")
    d = np.load('factorizations_f2.npz', allow_pickle=True)
    scheme = d['3,3,3']

    if isinstance(scheme, np.ndarray) and scheme.ndim == 0:
        scheme = scheme.item()

    if isinstance(scheme, dict):
        U_f2 = np.array(scheme['u'], dtype=np.int8)
        V_f2 = np.array(scheme['v'], dtype=np.int8)
        W_f2 = np.array(scheme['w'], dtype=np.int8)
    else:
        U_f2 = np.array(scheme[0], dtype=np.int8)
        V_f2 = np.array(scheme[1], dtype=np.int8)
        W_f2 = np.array(scheme[2], dtype=np.int8)

    print(f"Successfully loaded 3x3x3 F2 scheme: {U_f2.shape}\n")

    simulated_annealing_f2(U_f2, V_f2, W_f2, col_drop=0, iterations=25_000_000)

Loading F2 Factorizations...
Successfully loaded 3x3x3 F2 scheme: (9, 23)

Starting F2 Simulated Annealing | Dropping Col 0
Initial Errors: 46/729
Schedule: 25000000 steps | Temp: 2.0 -> 0.005

  Step         3 | Temp: 2.0000 | NEW BEST: 45/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      2429 | Temp: 1.9988 | NEW BEST: 43/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      2632 | Temp: 1.9987 | NEW BEST: 41/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      2842 | Temp: 1.9986 | NEW BEST: 40/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      3700 | Temp: 1.9982 | NEW BEST: 39/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      3711 | Temp: 1.9982 | NEW BEST: 36/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      3825 | Temp: 1.9982 | NEW BEST: 34/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      3946 | Temp: 1.9981 | NEW BEST: 32/729 errors
Saved checkpoint to sa_checkpoint.npz
  Step      4151 | Temp: 1.9980 | NEW 

# 1B

In [1]:
import os
from multiprocessing import Process, cpu_count
from sa_utils import run_chain

if __name__ == '__main__':
    out_dir = 'sa_runs'
    os.makedirs(out_dir, exist_ok=True)
    
    max_workers = max(1, cpu_count() - 2)
    cols = list(range(23))
    print(f"Starting SA chains for {len(cols)} configurations with {max_workers} workers...")
    
    for i in range(0, len(cols), max_workers):
        batch = cols[i:i+max_workers]
        processes = []
        for c in batch:
            p = Process(target=run_chain, args=(c, 25_000_000, out_dir))
            p.start()
            processes.append(p)
            
        for p in processes:
            p.join()
            
    print("All Phase 1B SA chains completed.")


Starting SA chains for 23 configurations with 10 workers...
All Phase 1B SA chains completed.


# 2 

In [ ]:
import numpy as np
import z3
import time
from functools import reduce

z3.set_param('parallel.enable', True)
z3.set_param('sat.threads', 0)
z3.set_param('smt.threads', 0)

def build_target_f2():
    T = np.zeros((9, 9, 9), dtype=int)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1
    return T

def solve_final_mile(U_best, V_best, W_best, max_flips_limit=15,
                      per_k_timeout_ms=5 * 60 * 1000):
    """
    Takes the SA checkpoint and searches for an exact 0-error state within
    an expanding Hamming radius. Only rows involved in current errors are
    declared as Z3 variables -- everything else is baked in as a constant,
    which lets Z3 constant-fold and drastically shrinks the search.
    """
    print("Initializing Z3 Solver Engine...")
    T_target = build_target_f2()
    rank = U_best.shape[1]

    def recon_f2(U, V, W):
        r = np.einsum("ir,jr,kr->ijk", U.astype(np.int32), V.astype(np.int32), W.astype(np.int32))
        return r % 2

    recon = recon_f2(U_best, V_best, W_best)
    mismatches = np.argwhere(recon != T_target)
    print(f"Checkpoint has {len(mismatches)} errors out of 729.")

    rows_U = sorted(set(int(m[0]) for m in mismatches))
    rows_V = sorted(set(int(m[1]) for m in mismatches))
    rows_W = sorted(set(int(m[2]) for m in mismatches))
    print(f"Free rows -> U:{rows_U}  V:{rows_V}  W:{rows_W}")

    def make_row_vars(name, free_rows, best_mat):
        row_vars = {}
        for i in range(9):
            if i in free_rows:
                row_vars[i] = [z3.Bool(f"{name}_{i}_{r}") for r in range(rank)]
            else:
                row_vars[i] = [bool(best_mat[i, r]) for r in range(rank)]
        return row_vars

    U_vars = make_row_vars("U", rows_U, U_best)
    V_vars = make_row_vars("V", rows_V, V_best)
    W_vars = make_row_vars("W", rows_W, W_best)

    print("Building algebraic constraints (skipping cells that can't change)...")
    mm_constraints = []
    free_i, free_j, free_k = set(rows_U), set(rows_V), set(rows_W)
    for i in range(9):
        for j in range(9):
            for k in range(9):
                if i not in free_i and j not in free_j and k not in free_k:
                    continue  # every factor here is a fixed constant -> already correct
                terms = [z3.And(U_vars[i][r], V_vars[j][r], W_vars[k][r]) for r in range(rank)]
                xor_sum = reduce(lambda x, y: z3.Xor(x, y), terms)
                mm_constraints.append(xor_sum == bool(T_target[i, j, k]))
    print(f"  {len(mm_constraints)} active constraints (skipped {729 - len(mm_constraints)} unreachable cells)")

    differences = []
    for i in rows_U:
        for r in range(rank):
            differences.append(U_vars[i][r] != bool(U_best[i, r]))
    for j in rows_V:
        for r in range(rank):
            differences.append(V_vars[j][r] != bool(V_best[j, r]))
    for k in rows_W:
        for r in range(rank):
            differences.append(W_vars[k][r] != bool(W_best[k, r]))
    print(f"  {len(differences)} free bits available to flip")

    for allowed_flips in range(1, max_flips_limit + 1):
        print(f"\n[Search Radius: {allowed_flips} bits] Solving...")
        solver = z3.Solver()
        solver.set("timeout", per_k_timeout_ms)
        solver.add(mm_constraints)
        solver.add(z3.AtMost(*differences, allowed_flips))

        start_t = time.time()
        result = solver.check()
        elapsed = time.time() - start_t

        if result == z3.sat:
            print(f"*** EXACT 0-ERROR SOLUTION FOUND in {elapsed:.2f}s at radius {allowed_flips}! ***")
            model = solver.model()

            def extract(row_vars, best_mat):
                out = best_mat.copy()
                for i, vars_or_const in row_vars.items():
                    for r in range(rank):
                        v = vars_or_const[r]
                        out[i, r] = int(v) if isinstance(v, bool) else (1 if z3.is_true(model[v]) else 0)
                return out

            U_perfect = extract(U_vars, U_best)
            V_perfect = extract(V_vars, V_best)
            W_perfect = extract(W_vars, W_best)
            return U_perfect, V_perfect, W_perfect

        elif result == z3.unsat:
            print(f"UNSAT: proven 0 errors is unreachable within {allowed_flips} flips of this checkpoint.")
        else:
            print(f"UNKNOWN: hit the {per_k_timeout_ms/1000:.0f}s timeout at radius {allowed_flips} ({elapsed:.1f}s elapsed).")

    print(f"\nNo 0-error state found within {max_flips_limit} bit flips of this checkpoint.")
    return None, None, None


if __name__ == "__main__":
    try:
        data = np.load('sa_checkpoint.npz')
        best_U, best_V, best_W = data['U'], data['V'], data['W']

        U_perf, V_perf, W_perf = solve_final_mile(
            best_U, best_V, best_W, max_flips_limit=15, per_k_timeout_ms=5 * 60 * 1000
        )

        if U_perf is not None:
            print(f"\nSaving perfect Rank-{best_U.shape[1]} factorization to 'perfect_rank.npz'...")
            np.savez('perfect_rank.npz', U=U_perf, V=V_perf, W=W_perf)

    except FileNotFoundError:
        print("Please run the SA phase first to generate 'sa_checkpoint.npz'.")

Initializing Z3 Solver Engine...
Checkpoint has 5 errors out of 729.
Free rows -> U:[0, 2, 3, 6, 7]  V:[0, 1, 2, 5, 7]  W:[1, 2, 3, 7, 8]
Building algebraic constraints (skipping cells that can't change)...
  665 active constraints (skipped 64 unreachable cells)
  330 free bits available to flip

[Search Radius: 1 bits] Solving...
UNSAT: proven 0 errors is unreachable within 1 flips of this checkpoint.

[Search Radius: 2 bits] Solving...
UNSAT: proven 0 errors is unreachable within 2 flips of this checkpoint.

[Search Radius: 3 bits] Solving...
UNSAT: proven 0 errors is unreachable within 3 flips of this checkpoint.

[Search Radius: 4 bits] Solving...
UNSAT: proven 0 errors is unreachable within 4 flips of this checkpoint.

[Search Radius: 5 bits] Solving...
UNSAT: proven 0 errors is unreachable within 5 flips of this checkpoint.

[Search Radius: 6 bits] Solving...
UNSAT: proven 0 errors is unreachable within 6 flips of this checkpoint.

[Search Radius: 7 bits] Solving...
UNSAT: proven

# 2B

In [1]:
import numpy as np
import z3
import time, glob, os, random
from functools import reduce

z3.set_param('parallel.enable', True)
z3.set_param('sat.threads', 0)
z3.set_param('smt.threads', 0)

# ==========================================
# Utility functions
# ==========================================
def build_target_f2():
    T = np.zeros((9, 9, 9), dtype=int)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1
    return T

def calc_f2_tensor(U, V, W):
    recon = np.einsum("ir,jr,kr->ijk", U.astype(np.int32), V.astype(np.int32), W.astype(np.int32))
    return (recon % 2).astype(np.int8)

def calc_error(U, V, W):
    recon = calc_f2_tensor(U, V, W)
    return int((recon != T_target).sum())

T_target = build_target_f2()

# ==========================================
# Quick probing: short SA runs around checkpoint
# ==========================================
def quick_probe(U_best, V_best, W_best, col_drop, probes=5, probe_steps=10_000):
    """
    Run very short SA iterations around the checkpoint, with small perturbations.
    Returns the best state found and its error count.
    """
    rank = U_best.shape[1]
    best_error_found = calc_error(U_best, V_best, W_best)
    best_state = (U_best.copy(), V_best.copy(), W_best.copy())

    print(f"  [Probe] Starting from error={best_error_found}, trying {probes} perturbations...")

    for probe_id in range(probes):
        U, V, W = U_best.copy(), V_best.copy(), W_best.copy()

        # Perturb by flipping ~3-5 random bits
        for _ in range(random.randint(3, 5)):
            mat = random.randint(0, 2)
            row = random.randint(0, 8)
            col = random.randint(0, rank - 1)
            if mat == 0:
                U[row, col] ^= 1
            elif mat == 1:
                V[row, col] ^= 1
            else:
                W[row, col] ^= 1

        # Run a short SA run from there
        recon = calc_f2_tensor(U, V, W)
        error_mask = (recon != T_target)
        current_error = int(error_mask.sum())
        violated = set(map(tuple, np.argwhere(error_mask)))

        T_start = 1.0
        T_end = 0.01
        cooling_factor = (T_end / T_start) ** (1.0 / probe_steps)
        temp = T_start

        for step in range(probe_steps):
            if not violated:
                break

            # Biased flip
            if violated and random.random() < 0.7:
                vi, vj, vk = random.choice(tuple(violated))
                r = random.randint(0, rank - 1)
                mat_choice = random.randint(0, 2)
                row = vi if mat_choice == 0 else (vj if mat_choice == 1 else vk)
                col = r
            else:
                mat_choice = random.randint(0, 2)
                row = random.randint(0, 8)
                col = random.randint(0, rank - 1)

            # Compute delta
            if mat_choice == 0:
                js = np.nonzero(V[:, col])[0]
                ks = np.nonzero(W[:, col])[0]
                idx_i = np.full(js.size * ks.size, row)
                idx_j = np.repeat(js, ks.size)
                idx_k = np.tile(ks, js.size)
            elif mat_choice == 1:
                is_ = np.nonzero(U[:, col])[0]
                ks = np.nonzero(W[:, col])[0]
                idx_i = np.repeat(is_, ks.size)
                idx_j = np.full(is_.size * ks.size, row)
                idx_k = np.tile(ks, is_.size)
            else:
                is_ = np.nonzero(U[:, col])[0]
                js = np.nonzero(V[:, col])[0]
                idx_i = np.repeat(is_, js.size)
                idx_j = np.tile(js, is_.size)
                idx_k = np.full(is_.size * js.size, row)

            touched = (idx_i, idx_j, idx_k)
            old_mismatch = error_mask[touched]
            new_recon_vals = 1 - recon[touched]
            new_mismatch = (new_recon_vals != T_target[touched])
            delta_E = int(new_mismatch.sum()) - int(old_mismatch.sum())

            accept = delta_E <= 0 or random.random() < math.exp(-delta_E / temp)

            if accept:
                if mat_choice == 0:
                    U[row, col] ^= 1
                elif mat_choice == 1:
                    V[row, col] ^= 1
                else:
                    W[row, col] ^= 1

                recon[touched] = new_recon_vals
                for (ii, jj, kk), was_wrong, is_wrong in zip(zip(*touched), old_mismatch, new_mismatch):
                    cell = (int(ii), int(jj), int(kk))
                    if was_wrong and not is_wrong:
                        violated.discard(cell)
                    elif not was_wrong and is_wrong:
                        violated.add(cell)
                error_mask[touched] = new_mismatch
                current_error += delta_E

            temp *= cooling_factor

        err = calc_error(U, V, W)
        print(f"    Probe {probe_id + 1}/{probes}: error={err}")
        if err < best_error_found:
            best_error_found = err
            best_state = (U.copy(), V.copy(), W.copy())

    print(f"  [Probe] Best found: error={best_error_found}")
    return best_state, best_error_found


# ==========================================
# Z3 Final-Mile Solver (with multi-seed per radius)
# ==========================================
def solve_final_mile(U_best, V_best, W_best, max_flips_limit=15,
                      per_k_timeout_ms=5 * 60 * 1000, seeds_per_radius=3):
    """
    Takes the SA checkpoint and searches for an exact 0-error state within
    an expanding Hamming radius. Only rows involved in current errors are
    declared as Z3 variables. Tries multiple random seeds per radius.
    """
    T_target = build_target_f2()
    rank = U_best.shape[1]

    def recon_f2(U, V, W):
        r = np.einsum("ir,jr,kr->ijk", U.astype(np.int32), V.astype(np.int32), W.astype(np.int32))
        return r % 2

    recon = recon_f2(U_best, V_best, W_best)
    mismatches = np.argwhere(recon != T_target)
    if len(mismatches) == 0:
        print("  Checkpoint already has 0 errors!")
        return U_best, V_best, W_best

    rows_U = sorted(set(int(m[0]) for m in mismatches))
    rows_V = sorted(set(int(m[1]) for m in mismatches))
    rows_W = sorted(set(int(m[2]) for m in mismatches))
    print(f"  Free rows: U={rows_U}, V={rows_V}, W={rows_W}")

    def make_row_vars(name, free_rows, best_mat):
        row_vars = {}
        for i in range(9):
            if i in free_rows:
                row_vars[i] = [z3.Bool(f"{name}_{i}_{r}") for r in range(rank)]
            else:
                row_vars[i] = [bool(best_mat[i, r]) for r in range(rank)]
        return row_vars

    U_vars = make_row_vars("U", rows_U, U_best)
    V_vars = make_row_vars("V", rows_V, V_best)
    W_vars = make_row_vars("W", rows_W, W_best)

    print("  Building algebraic constraints...")
    mm_constraints = []
    free_i, free_j, free_k = set(rows_U), set(rows_V), set(rows_W)
    for i in range(9):
        for j in range(9):
            for k in range(9):
                if i not in free_i and j not in free_j and k not in free_k:
                    continue
                terms = [z3.And(U_vars[i][r], V_vars[j][r], W_vars[k][r]) for r in range(rank)]
                xor_sum = reduce(lambda x, y: z3.Xor(x, y), terms)
                mm_constraints.append(xor_sum == bool(T_target[i, j, k]))
    print(f"  {len(mm_constraints)} active constraints")

    differences = []
    for i in rows_U:
        for r in range(rank):
            differences.append(U_vars[i][r] != bool(U_best[i, r]))
    for j in rows_V:
        for r in range(rank):
            differences.append(V_vars[j][r] != bool(V_best[j, r]))
    for k in rows_W:
        for r in range(rank):
            differences.append(W_vars[k][r] != bool(W_best[k, r]))
    print(f"  {len(differences)} free bits")

    for allowed_flips in range(1, max_flips_limit + 1):
        per_seed_timeout = per_k_timeout_ms // seeds_per_radius
        print(f"\n  [Radius {allowed_flips}] Trying {seeds_per_radius} seeds "
              f"({per_seed_timeout}ms each)...")

        for seed_offset in range(seeds_per_radius):
            solver = z3.Solver()
            solver.set("timeout", per_seed_timeout)

            # Tune Z3 for dense algebraic formulas
            solver.set("random_seed", 42 + allowed_flips * 100 + seed_offset)
            solver.set("max_conflicts", 10_000_000)

            solver.add(mm_constraints)
            solver.add(z3.AtMost(*differences, allowed_flips))

            start_t = time.time()
            result = solver.check()
            elapsed = time.time() - start_t

            status_str = str(result)
            print(f"    seed {seed_offset + 1}/{seeds_per_radius}: {status_str} ({elapsed:.1f}s)")

            if result == z3.sat:
                print(f"  *** SAT FOUND at radius {allowed_flips}, seed {seed_offset + 1}! ***")
                model = solver.model()

                def extract(row_vars, best_mat):
                    out = best_mat.copy()
                    for i, vars_or_const in row_vars.items():
                        for r in range(rank):
                            v = vars_or_const[r]
                            out[i, r] = int(v) if isinstance(v, bool) else (1 if z3.is_true(model[v]) else 0)
                    return out

                return extract(U_vars, U_best), extract(V_vars, V_best), extract(W_vars, W_best)

    return None, None, None


# ==========================================
# Main execution
# ==========================================
# ==========================================
# Main execution (Phase 2B FIXED)
# ==========================================
if __name__ == "__main__":
    import math
    import glob
    import os
    import numpy as np

    # Use absolute paths to guarantee Jupyter finds the folder
    search_path = os.path.abspath(os.path.join("sa_runs", "sa_checkpoint_drop*.npz"))
    checkpoints = sorted(glob.glob(search_path))
    
    # [FIX]: Replaced exit() with an Exception to prevent kernel crashes
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoints found in {os.path.abspath('sa_runs')} -- run Phase 1b first.")

    print("=== PHASE 2B: Z3 SWEEP WITH PROBING ===\n")

    # Rank by fewest errors
    summaries = []
    for path in checkpoints:
        data = np.load(path)
        summaries.append((int(data['error']), int(data['col_drop']), path))
    summaries.sort()

    print("Checkpoint ranking (best first):")
    for err, cd, path in summaries:
        print(f"  col_drop={cd:2d}  errors={err:3d}  {path}")

    print("\n" + "=" * 60)

    # Try the best 5 checkpoints with probing, then Z3
    for rank_idx, (err, cd, path) in enumerate(summaries[:5]):
        print(f"\n[Rank {rank_idx + 1}/5] col_drop={cd}, baseline errors={err}")
        print(f"  Loading {path}...")
        data = np.load(path)
        U_best, V_best, W_best = data['U'], data['V'], data['W']

        # Probe this checkpoint to see if nearby state is better
        print(f"  Probing neighborhood...")
        (U_best, V_best, W_best), probed_err = quick_probe(U_best, V_best, W_best, cd, probes=5, probe_steps=10_000)

        if probed_err == 0:
            print(f"\n*** FOUND 0-ERROR DURING PROBING AT col_drop={cd}! ***")
            np.savez(f"perfect_rank22_drop{cd:02d}.npz", U=U_best, V=V_best, W=W_best, col_drop=cd)
            break

        # Now try Z3
        print(f"\n  Launching Z3 solver (max radius 15, 3 seeds per radius)...")
        U_perf, V_perf, W_perf = solve_final_mile(
            U_best, V_best, W_best, max_flips_limit=15, 
            per_k_timeout_ms=5 * 60 * 1000, seeds_per_radius=3
        )

        if U_perf is not None:
            out_path = f"perfect_rank22_drop{cd:02d}.npz"
            np.savez(out_path, U=U_perf, V=V_perf, W=W_perf, col_drop=cd)
            print(f"\n*** SUCCESS! 0-ERROR SOLUTION SAVED TO {out_path} ***")
            break
    else:
        print("\n" + "=" * 60)
        print("No 0-error solution found in top 5 checkpoints.")

=== PHASE 2B: Z3 SWEEP WITH PROBING ===

Checkpoint ranking (best first):
  col_drop= 2  errors=  5  c:\Users\Prakshil\OneDrive\Desktop\IITK\Tensor-Rank-Optimization-depth-3-matrices\algorithms\sa_runs\sa_checkpoint_drop02.npz
  col_drop= 3  errors=  5  c:\Users\Prakshil\OneDrive\Desktop\IITK\Tensor-Rank-Optimization-depth-3-matrices\algorithms\sa_runs\sa_checkpoint_drop03.npz
  col_drop= 5  errors=  5  c:\Users\Prakshil\OneDrive\Desktop\IITK\Tensor-Rank-Optimization-depth-3-matrices\algorithms\sa_runs\sa_checkpoint_drop05.npz
  col_drop= 8  errors=  5  c:\Users\Prakshil\OneDrive\Desktop\IITK\Tensor-Rank-Optimization-depth-3-matrices\algorithms\sa_runs\sa_checkpoint_drop08.npz
  col_drop= 9  errors=  5  c:\Users\Prakshil\OneDrive\Desktop\IITK\Tensor-Rank-Optimization-depth-3-matrices\algorithms\sa_runs\sa_checkpoint_drop09.npz
  col_drop=10  errors=  5  c:\Users\Prakshil\OneDrive\Desktop\IITK\Tensor-Rank-Optimization-depth-3-matrices\algorithms\sa_runs\sa_checkpoint_drop10.npz
  col_dr